In [4]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Parámetros clave
CSV_PATH = "../data/staging/operaciones_semana_2025-04-27_2025-05-03.csv"
TIME_BUDGET = 120   # segundos para AutoML
SEED = 42
OUTDIR = Path("outputs_flaml_reg")
OUTDIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)


In [5]:
# === 1. Carga de datos ===
df = pd.read_csv(CSV_PATH)
TARGET = "precioTN"
FEATURES = [
    "tipoOperacion", "tipoModalidad", "tipoOiv", "grano", "volumenTN",
    "condicionCalidad", "procedenciaProvincia", "lugarEntrega",
    "condicionPago", "esDestinoFinal", "cosecha"
]
df = df.dropna(subset=[TARGET]).copy()

# === 2. Preprocesamiento simple ===
for col in FEATURES:
    if col not in df.columns:
        df[col] = np.nan
    if df[col].dtype == "object":
        df[col] = df[col].astype(str)

cat_cols = [c for c in FEATURES if df[c].dtype == "object"]
num_cols = [c for c in FEATURES if c not in cat_cols]

X_num = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = pd.get_dummies(df[cat_cols], drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = pd.to_numeric(df[TARGET], errors="coerce")

feature_columns = X.columns.tolist()

# === 3. Split train/test ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# === 4. Entrenamiento con FLAML ===
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="regression",
    time_budget=TIME_BUDGET,
    metric="rmse",
    seed=SEED,
    verbose=1
)

print("\n=== Mejor modelo ===")
print("Estimador:", automl.best_estimator)
print("Pérdida (RMSE interno):", automl.best_loss)
print("Configuración:")
print(json.dumps(automl.best_config, indent=2))

# === 5. Evaluación ===
y_pred = automl.predict(X_test)
# Compatibilidad con versiones viejas de sklearn
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== Métricas en test ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

# === 6. Guardado de artefactos ===
pd.DataFrame({"y_true": y_test, "y_pred": y_pred}).to_csv(OUTDIR / "predicciones_test.csv", index=False)
X_train.assign(precioTN=y_train).to_csv(OUTDIR / "train_processed.csv", index=False)
X_test.assign(precioTN=y_test).to_csv(OUTDIR / "test_processed.csv", index=False)
with open(OUTDIR / "feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

summary = {
    "metrics_test": {"rmse": rmse, "mae": mae, "r2": r2},
    "n_train": len(X_train),
    "n_test": len(X_test),
    "best_estimator": automl.best_estimator,
    "best_loss": automl.best_loss,
    "best_config": automl.best_config
}
with open(OUTDIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

joblib.dump(automl, OUTDIR / "best_automl.pkl")

print("\nArtefactos guardados en:", OUTDIR.resolve())



=== Mejor modelo ===
Estimador: extra_tree
Pérdida (RMSE interno): 73123.98503558667
Configuración:
{
  "n_estimators": 371,
  "max_features": 0.26809588181602056,
  "max_leaves": 65
}

=== Métricas en test ===
RMSE: 70899.4310
MAE:  38654.0237
R2:   0.5934

Artefactos guardados en: /Users/paul/Downloads/Documentos Locales/web-mining-sio-granos/notebooks/outputs_flaml_reg
